In [2]:
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from PIL import Image
from torchvision import transforms
import pandas as pd
import torch
import matplotlib.pyplot as plt

In [5]:
class CaptchaDataset(Dataset):
    def __init__(self, image_dir, csv_path, transform=None, max_length=10):
        # Store paths and settings
        self.image_dir = Path(image_dir)
        self.transform = transform
        self.max_length = max_length

        # Load label table once at startup
        self.labels_df = pd.read_csv(csv_path)

        # Extract filenames and text labels from the CSV
        self.image_names = self.labels_df["filename"].tolist()
        self.labels = self.labels_df["label"].tolist()

        # Define allowed characters and add one blank token for padding
        self.chars = "2346789ABCDEFGHJKLMNPQRTUVWXYabcdefghijkmnpqrtuvwxy"
        self.blank_token = "<BLANK>"

        # Create character-to-index mapping
        self.char_to_idx = {c: i for i, c in enumerate(self.chars)}
        self.char_to_idx[self.blank_token] = len(self.chars)

        # Create reverse mapping for debugging / decoding
        self.idx_to_char = {i: c for c, i in self.char_to_idx.items()}

    def __len__(self):
        # Return total number of samples
        return len(self.image_names)

    def encode_label(self, label):
        # Turn text label into list of integer indices
        encoded = [self.char_to_idx[c] for c in label]

        # Pad shorter labels up to max_length using blank token
        while len(encoded) < self.max_length:
            encoded.append(self.char_to_idx[self.blank_token])

        return torch.tensor(encoded, dtype=torch.long)

    def decode_label(self, encoded_tensor):
        # Convert encoded tensor back to readable text, ignoring blanks
        chars = []
        blank_idx = self.char_to_idx[self.blank_token]

        for idx in encoded_tensor.tolist():
            if idx != blank_idx:
                chars.append(self.idx_to_char[idx])

        return "".join(chars)

    def __getitem__(self, idx):
        # Get filename and raw label for one sample
        img_name = self.image_names[idx]
        label = self.labels[idx]

        # Load grayscale image
        img_path = self.image_dir / img_name
        image = Image.open(img_path).convert("L")

        # Apply transforms such as ToTensor()
        if self.transform:
            image = self.transform(image)

        # Encode the label into fixed-length numeric form
        label_tensor = self.encode_label(label)

        return image, label_tensor